# Introduction to Data Analysis with Python
### COVID-19 Global Dataset · Publication-Quality Figures

---

**Workshop outline (45 min)**

| # | Section | Time |
|---|---------|------|
| 0 | Setup | 2 min |
| 1 | Load & prepare data | 5 min |
| 2 | Explore the data | 5 min |
| 3a | Violin + box chart | 8 min |
| 3b | Grouped bar chart | 8 min |
| 3c | Composite 2-panel figure | 7 min |

---

**Dataset source:** [Our World in Data — COVID-19](https://github.com/owid/covid-19-data) — daily pandemic statistics for 240+ countries, 2020–2024. Variables include smoothed death rates, vaccination coverage, and GDP per capita.

Each row represents **one country on one day**. We aggregate to country-level annual means and ask: *how did mortality differ by continent and income group, and did that gap change over time?*

**Today's goal:** Build figures that look like they belong in a journal article — applying design conventions from a real *Journal of Bacteriology* paper to answer real epidemiological questions.

---
## Figure Style Reference

The figures in this notebook are modelled on **Wang et al., *Journal of Bacteriology*, 2024** ([PMID 39329528](https://pubmed.ncbi.nlm.nih.gov/39329528/)) — a paper on mRNA localisation in cyanobacteria. Its figures follow conventions standard across biomedical journals:

| Design choice | What it communicates |
|---------------|---------------------|
| **Bar height = mean** | Central tendency of the group |
| **Error bar = SEM** | Precision of the mean estimate |
| **Individual dots overlaid** | Raw data transparency — lets readers judge spread |
| **Violin shape** | Full distribution — skewness, bimodality, outliers |
| **Multi-panel (A)(B)** | One figure tells a coherent, multi-part story |
| **No top/right spines** | Clean, uncluttered — `sns.despine()` |
| **`dpi=300` on save** | Meets most journal submission requirements |

> **Key principle:** Every visual element should earn its place. Add individual dots over bars so you don't hide your data. Remove gridlines if they don't help the reader.

---
## 0. Setup  ⏱ 2 min

| Library | What it does |
|---------|-------------|
| `pandas` | Load and manipulate tabular data |
| `numpy` | Numerical arrays and maths |
| `matplotlib` | Base plotting engine |
| `seaborn` | High-level statistical plots |
| `scipy` | Statistics — hypothesis tests, regression |

All pre-installed in Google Colab.

In [1]:
!pip install pandas matplotlib seaborn scipy numpy --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, kruskal, linregress

# Publication style — modelled on Wang et al., J. Bacteriol. 2024
plt.rcParams.update({
    "font.family"      : "sans-serif",
    "font.size"        : 10,
    "axes.linewidth"   : 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size" : 3.5,
    "ytick.major.size" : 3.5,
    "figure.dpi"       : 120,
    "savefig.dpi"      : 300,
})
sns.set_theme(style="white", context="paper", font_scale=1.15)

# ── Helpers (style: plotivy.app/blog/biology-statistics-figures-python) ──────

def sig_stars(p):
    """Convert p-value to * notation."""
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def draw_bracket(ax, x1, x2, y, text, lw=1.2,
                 color="black", fontsize=10):
    """
    Simple significance line (no brackets/ticks).
    """
    ax.plot([x1, x2], [y, y], lw=lw, color=color, clip_on=False)
    ax.text((x1 + x2) / 2, y, text,
            ha="center", va="bottom", fontsize=fontsize, color=color)

print("Libraries loaded. Publication style applied.")

---
## 1. Load & Prepare Data  ⏱ 5 min

The dataset lives on GitHub — we load it directly from the URL using `pd.read_csv()`, no download needed. Loading may take 20–30 s in Colab.

> **How to get a raw CSV URL from any GitHub repo:**  
> 1. Navigate to the file in the repo  
> 2. Click the **Raw** button (top-right of the file viewer)  
> 3. Copy the URL from the browser address bar — that's what you pass to `pd.read_csv()`

In [ ]:
COLS = [
    "continent", "location", "date",
    "new_deaths_smoothed_per_million",
    "people_fully_vaccinated_per_hundred",
    "gdp_per_capita",
]
URL = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"

print("Loading data (~30 s in Colab)…")
df_raw = pd.read_csv(URL, usecols=COLS, parse_dates=["date"])
df     = df_raw[df_raw["continent"].notna()].reset_index(drop=True)
print(f"Loaded: {len(df):,} rows · {df['location'].nunique()} countries")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")

In [ ]:
# ── Annual per-country mean deaths ──────────────────────────────────────────
df_ann = (
    df.query("new_deaths_smoothed_per_million >= 0")
      .assign(year=lambda x: x["date"].dt.year.astype(str))
      .groupby(["location", "continent", "year"])["new_deaths_smoothed_per_million"]
      .mean()
      .reset_index(name="avg_deaths")
      .dropna()
)

# ── Income groups from GDP tertiles ─────────────────────────────────────────
gdp_map   = df.groupby("location")["gdp_per_capita"].first().dropna()
q33, q67  = gdp_map.quantile(0.33), gdp_map.quantile(0.67)

def _income(g):
    if g < q33: return "Low income"
    if g < q67: return "Middle income"
    return "High income"

df_ann["income_group"] = df_ann["location"].map(gdp_map.apply(_income))

# ── Convenience subsets ─────────────────────────────────────────────────────
df_2021 = df_ann[df_ann["year"] == "2021"].dropna(subset=["avg_deaths"])
y21     = df_ann[df_ann["year"] == "2021"]["avg_deaths"].dropna()
y23     = df_ann[df_ann["year"] == "2023"]["avg_deaths"].dropna()

# ── 2022 vaccination vs mortality (for scatter panel) ───────────────────────
df_scat = (
    df[df["date"].dt.year == 2022]
    .groupby(["location", "continent"])
    .agg(max_vacc   = ("people_fully_vaccinated_per_hundred", "max"),
         avg_deaths = ("new_deaths_smoothed_per_million",     "mean"))
    .reset_index()
    .query("max_vacc > 1 and avg_deaths >= 0")
    .dropna()
)

print(f"df_ann  : {len(df_ann):,} rows | {df_ann['location'].nunique()} countries × years")
print(f"df_2021 : {len(df_2021)} country rows")
print(f"df_scat : {len(df_scat)} countries (2022)")
print(f"y21 n={len(y21)},  y23 n={len(y23)}")

---
## 2. Explore the Data  ⏱ 5 min

Before plotting, spend a few minutes understanding the structure and content of your dataset.

| Command | What it tells you |
|---------|-------------------|
| `df.shape` | Rows × columns |
| `df.columns` | Column names |
| `df.info()` | Data types + non-null counts per column |
| `df.head()` | First 5 rows |
| `df.describe()` | Summary statistics for numeric columns |
| `df["col"].value_counts()` | Frequency of each category |

> **Why this matters:** Missing values (`NaN`) are common in real datasets. `df.info()` shows you exactly which columns have gaps — critical before computing means or running statistical tests.

In [ ]:
# Dataset structure
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print()
df.info()

In [ ]:
# Data preview and summary
display(df.head())

print("\n--- Rows per continent ---")
print(df["continent"].value_counts())

print("\n--- Numeric summary ---")
display(df[["new_deaths_smoothed_per_million",
            "people_fully_vaccinated_per_hundred",
            "gdp_per_capita"]].describe().round(2))

---
## 3. Publication-Style Figures  ⏱ 23 min

| Figure | Type | Scientific question |
|--------|------|---------------------|
| 3a | **Violin + box** | Which continent had the highest COVID-19 mortality in 2021? |
| 3b | **Grouped bar** | Did the income–mortality gap narrow as the pandemic progressed? |
| 3c | **Composite 2-panel** | Tell both stories in one publication-ready figure |

**New Python techniques introduced:**
- `sns.violinplot(inner="box")` — distribution shape + IQR box in one layer
- `ax.errorbar(fmt="D")` — mean ± SEM overlaid as diamond marker
- Jittered `ax.scatter()` — individual data points overlaid on bars
- `kruskal()` / `mannwhitneyu()` — non-parametric multi-group tests
- `np.arange()` + manual offsets — grouped bar layout
- `plt.subplots(1, 2, gridspec_kw=...)` — composite panel figures

### 3a. Which continent had the highest mortality in 2021?  *(Violin + box plot with error bar)*

A **violin + box plot** combines three complementary views in one panel:
- **Violin** — kernel density estimate of the full distribution; width shows where data is concentrated
- **Box inside** — median (line), interquartile range (box body), and 1.5 × IQR whiskers
- **Mean ± SEM** — diamond marker + error bar overlaid on top

This is more informative than a bar chart: the violin reveals *shape* (skewness, bimodality), the box gives the *median and quartiles*, and the error bar shows *precision of the mean*.

> **`inner="box"`** in `sns.violinplot` draws the IQR box inside the violin automatically.  
> **`cut=0`** prevents the violin extending beyond the observed data range.  
> **`set_alpha()`** on `ax.collections` makes violin bodies semi-transparent so the inner box remains legible.

In [ ]:
CONTINENTS = ["Africa", "Asia", "Europe", "North America", "Oceania", "South America"]
PAL = dict(zip(CONTINENTS, sns.color_palette("Set2", 6)))
H, p_kw = kruskal(*[df_2021[df_2021["continent"] == c]["avg_deaths"].values for c in CONTINENTS])

def plot_3a(ax):
    sns.violinplot(data=df_2021[df_2021["continent"].isin(CONTINENTS)],
                   x="continent", y="avg_deaths", order=CONTINENTS,
                   hue="continent", palette=PAL, legend=False,
                   inner="box", cut=0, linewidth=0.8, ax=ax)
    for coll in ax.collections:
        coll.set_alpha(0.55)
    ax.set_ylim(ax.get_ylim())
    for i, cont in enumerate(CONTINENTS):
        vals = df_2021[df_2021["continent"] == cont]["avg_deaths"].dropna()
        ax.errorbar(i, vals.mean(), yerr=vals.sem(), fmt="D",
                    color="black", markersize=5, capsize=4, capthick=1.2,
                    linewidth=1.4, zorder=5, label="Mean ± SEM" if i == 0 else "")
    ax.set_xticks(range(len(CONTINENTS)))
    ax.set_xticklabels(CONTINENTS, rotation=25, ha="right", fontsize=9)
    ax.set_ylabel("Deaths / million / day", fontsize=9)
    ax.legend(frameon=False, fontsize=9, loc="upper right")
    sns.despine(ax=ax)

# Standalone plot
fig, ax = plt.subplots(figsize=(8, 5.5))
plot_3a(ax)
ax.set_title(f"COVID-19 Mortality by Continent — 2021\n"
             f"Kruskal–Wallis  H = {H:.1f},  p = {p_kw:.2e}", fontsize=10, loc="left")
plt.tight_layout()
plt.savefig("fig4a_violin_continent.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"n countries: {len(df_2021)}")
for c in CONTINENTS:
    v = df_2021[df_2021["continent"] == c]["avg_deaths"]
    print(f"  {c:<18}  n={len(v)}  median={v.median():.3f}  mean={v.mean():.3f}  SEM={v.sem():.3f}")

> **Reading the figure:**
> - **Violin width** at any height = relative density of countries at that mortality level. A wide violin near a value means many countries had that rate.
> - **Box** = IQR (25th–75th percentile); **red line** inside box = median; **whiskers** = 1.5 × IQR.
> - **Diamond** = mean; **error bar** = SEM. When mean (diamond) sits above median (red line), the distribution is right-skewed — a few high-mortality outliers pull the mean up.
> - **South America** shows a wide violin at high values — a small number of countries had very high mortality, dragging the mean above the median.
> - **Africa** has a narrow violin near zero — most African countries reported very few deaths, though under-reporting is a known issue.
> - **Kruskal–Wallis p < 0.001** confirms significant differences among continents; a post-hoc Dunn test would identify which pairs differ.

### 3b. Did the income–mortality gap narrow over time?  *(Grouped bar chart)*

A **grouped bar chart** places multiple bars at each x-position — one per subgroup.  
Here: three income groups (from GDP tertiles) × three years.

This addresses a structural equity question: as vaccines became available, did they reach low-income countries fast enough to close the mortality gap?

> **Income classification:** We split countries into tertiles of GDP per capita — *not* the official World Bank income classification. Treat this as illustrative.

In [ ]:
u

In [ ]:
YEARS   = ["2021", "2022", "2023"]
GROUPS  = ["Low income", "Middle income", "High income"]
GCOLORS = {"Low income": "#e74c3c", "Middle income": "#f0a500", "High income": "#2980b9"}

df_inc    = df_ann[df_ann["year"].isin(YEARS) & df_ann["income_group"].notna()].copy()
stats_inc = (
    df_inc.groupby(["income_group", "year"])["avg_deaths"]
    .agg(mean="mean", sem=lambda x: x.std(ddof=1) / np.sqrt(len(x)), n="count")
    .reset_index()
)

def plot_3b(ax):
    w, x = 0.22, np.arange(len(YEARS))
    offsets = {grp: (gi - 1) * (w + 0.03) for gi, grp in enumerate(GROUPS)}

    for gi, grp in enumerate(GROUPS):
        off  = offsets[grp]
        sub  = stats_inc[stats_inc["income_group"] == grp].set_index("year")
        means = [sub.loc[yr, "mean"] if yr in sub.index else np.nan for yr in YEARS]
        sems  = [sub.loc[yr, "sem"]  if yr in sub.index else 0      for yr in YEARS]
        ax.bar(x + off, means, width=w, label=grp, color=GCOLORS[grp],
               edgecolor="black", linewidth=0.8, alpha=0.85, zorder=2)
        ax.errorbar(x + off, means, yerr=sems, fmt="none",
                    color="black", capsize=4, capthick=1.2, linewidth=1.3, zorder=4)

    # Per-year local tops; fixed gap anchored to global scale
    yr_tops = {yr: float((stats_inc[stats_inc["year"] == yr]["mean"] +
                          stats_inc[stats_inc["year"] == yr]["sem"]).max())
               for yr in YEARS}
    data_top = max(yr_tops.values())
    gap = data_top * 0.12
    ax.set_ylim(0, data_top * 1.60)

    # Three brackets stacked at fixed multiples of gap above each year's local top
    # (multipliers guarantee distinct heights regardless of which group has the tallest bar)
    PAIRS = [("Low income",    "Middle income", 1.0),
             ("Middle income", "High income",   1.8),
             ("Low income",    "High income",   2.8)]
    for yi, yr in enumerate(YEARS):
        for g1, g2, mult in PAIRS:
            v1 = df_inc[(df_inc["income_group"] == g1) & (df_inc["year"] == yr)]["avg_deaths"]
            v2 = df_inc[(df_inc["income_group"] == g2) & (df_inc["year"] == yr)]["avg_deaths"]
            if len(v1) > 1 and len(v2) > 1:
                _, p = mannwhitneyu(v1, v2, alternative="two-sided")
                draw_bracketl(ax, x[yi] + offsets[g1], x[yi] + offsets[g2],
                             yr_tops[yr] + gap * mult, sig_stars(p))

    ax.set_xticks(np.arange(len(YEARS)))
    ax.set_xticklabels(YEARS, fontsize=9)
    ax.set_xlabel("Year", fontsize=9)
    ax.set_ylabel("Deaths / million / day", fontsize=9)
    ax.legend(frameon=False, fontsize=9, loc="upper right")
    sns.despine(ax=ax)

# Standalone plot
fig, ax = plt.subplots(figsize=(8.5, 5.5))
plot_3b(ax)
ax.set_title("COVID-19 Mortality by Income Group (2021–2023)\n"
             "Brackets: all pairwise comparisons (Mann–Whitney)", fontsize=10, loc="left")
plt.tight_layout()
plt.savefig("fig4b_grouped_bar.pdf", dpi=300, bbox_inches="tight")
plt.show()

> **Reading the figure:**
> - **High-income countries** report higher mortality than low-income ones across all years — partly because wealthier countries have older populations and better death recording.
> - The **individual dots** show that within each income group there is wide country-level heterogeneity — averages hide important variation.
> - **Inner brackets** (Low–Middle, Middle–High) test adjacent group differences; the **outer bracket** (Low–High) tests the extremes.
> - Significance that persists into 2023 suggests the mortality gap did not fully close despite declining overall deaths.
> - By 2023 all bars converge toward zero, but the *relative* ordering of income groups is informative for equity discussions.

### 3c. Composite figure — two panels, one story

Because 3a and 3b wrap their plotting logic in `plot_3a(ax)` and `plot_3b(ax)`, combining them into a single publication figure takes just a few lines: create a figure with two axes, call each function, add panel labels and a shared title.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"wspace": 0.45})

plot_3a(axes[0])
plot_3b(axes[1])

axes[0].set_title("(A)  By continent, 2021",        fontsize=9, loc="left")
axes[1].set_title("(B)  By income group, 2021–2023", fontsize=9, loc="left")
fig.suptitle("COVID-19 Mortality — Continent & Income Group", fontsize=12, y=1.02)
plt.savefig("fig4c_composite.png", dpi=300, bbox_inches="tight")
plt.show()

---
## Key Takeaways

| What we built | Python technique | Design principle |
|---------------|-----------------|------------------|
| Violin + box (continent, 2021) | `sns.violinplot(inner="box")` + `ax.errorbar()` | Show distribution shape, not just mean |
| Grouped bar (income × year) | `np.arange()` + manual offset per group | Multi-group comparison at a glance |
| Composite 2-panel | `plt.subplots(1, 2, gridspec_kw=...)` | One figure, one coherent story |

**Publication checklist:**
1. `sns.despine()` — remove top and right spines
2. `dpi=300` on `savefig` — journal resolution requirement
3. Consistent font sizes across all panels (`fontsize=` set explicitly)
4. Individual data points overlaid on all bar charts
5. Error bars labelled as SEM or SD in the figure caption
6. Significance annotations only when the test is central to the message

> **The violin + box, grouped bar, and composite panel are among the most common figure types in biomedical papers. Mastering them in Python means you can generate publication-quality figures directly from your analysis pipeline.**

---
## Resources

| Resource | What it covers |
|----------|---------------|
| [Scientific Visualization: Python + Matplotlib](https://github.com/rougier/scientific-visualization-book) | Free book: publication-quality figures in Python |
| [statannotations](https://github.com/trevismd/statannotations) | Automated significance brackets on seaborn plots |
| [Wang et al., *J. Bacteriol.* 2024](https://pubmed.ncbi.nlm.nih.gov/39329528/) | Style reference paper for this notebook |
| [Nature methods — Points of Significance](https://www.nature.com/collections/qghhqm/pointsofsignificance) | Statistics for biologists — free article series |

### Figure format cheatsheet

```python
plt.savefig("fig.png",  dpi=300, bbox_inches="tight")   # vector — ideal for journals
plt.savefig("fig.pdf",  dpi=300, bbox_inches="tight")   # editable in Inkscape / Illustrator
plt.savefig("fig.tiff", dpi=600, bbox_inches="tight")   # high-res raster for submission
```

---
*Data: Our World in Data — ourworldindata.org/coronavirus*  
*Style reference: Wang et al., Journal of Bacteriology, October 2024, PMID 39329528*